In [ ]:
import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle as pk
from IPython.display import Image, display
import os
import sys
from sklearn import decomposition


In [ ]:
main_dir = os.getcwd()
if "notebooks" in main_dir:
    main_dir = main_dir[:-10]
elif main_dir == "/content":
    main_dir = main_dir + "/GPA-source_code"
os.chdir(main_dir)
print(main_dir)

contrast_colors = {
    0: '#C71585', #pink
    1: '#1f77b4',  # blue
    2: '#2ca02c',  # green
    3: '#ff7f0e',  # orange
    4: '#8c564b',  # brown
    5: '#d62728',  # red 
    6: '#9467bd'  # purple (to be used for index 8)
}

data_dir = main_dir + '/data/'
#result_dir = '%s/assets/Transport_genes/' % main_dir


In [ ]:

#loads weights, biases and parameters of trained velocity field
def load_W(filename):
    with open(filename, "rb") as fr:
        W, b, p = pk.load(fr)

    W = [tf.Variable(w,  dtype=tf.float32) for w in W]
    b = [tf.Variable(b_,  dtype=tf.float32) for b_ in b]

    return W, b, p

    
def v(x, t, W, b):   # neural newtork for time-dependent vectorfield
    num_layers = len(W)
    activation_ftn = tf.nn.tanh
        
    h = tf.concat([x, t*tf.ones([x.shape[0], 1], dtype=tf.float32)], axis=1)
    for l in range(0,num_layers-1):
        h = activation_ftn(tf.add(tf.matmul(h, W[l]), b[l]))
    out=tf.add(tf.matmul(h, W[-1]), b[-1])

    return out

#returns a list of positions over time
def time_integration(x0, T, dt):
    x = tf.constant(x0, dtype=tf.float32)
    xs = [x0]
    for i in range(int(T/dt)):
        vv = v(x, dt*i, W, b)
        x += dt * vv
        xs.append(x.numpy())
    return xs



In [ ]:
from scripts.util.downstream import gene_dynamics_whole_saveonly
from scripts.util.downstream import Average_gene_dynamics_whole_saveonly
from scripts.util.downstream import Average_gene_dynamics_whole_saveonly_with_violin_plot_sample1_EMT
from scripts.util.downstream import Average_gene_dynamics_whole_saveonly_with_violin_plot_sample_3_stem
from scripts.util.downstream import create_pdf_from_gene_images
from scripts.util.downstream import Compute_and_Plot_FoldChange_MeanDiff_PValues
from scripts.util.downstream import difference_of_means_emt
from scripts.util.downstream import difference_of_means_stem
from scripts.util.downstream import Average_gene_dynamics_whole_saveonly_single_trajectory_EMT
from scripts.util.downstream import Average_gene_dynamics_whole_saveonly_single_trajectory_mESC
from scripts.util.downstream import Compare_Distribution_Trajectories_Intermediate_EMT
from scripts.util.downstream import Compare_Distribution_Trajectories_Intermediate_mESC

In [ ]:
from scripts.util.downstream import plot_X1_hat_displacement_distribution
from scripts.util.downstream import generate_static_cluster_plot_deviation_colormap_MCF7
from scripts.util.downstream import generate_static_cluster_plot_deviation_colormap_PA3
from scripts.util.downstream import generate_static_cluster_plot_deviation_colormap_862
from scripts.util.downstream import generate_static_cluster_plot_deviation_colormap_887
from scripts.util.downstream import Average_gene_dynamics_whole_saveonly_single_trajectory_NDPR_breast_cancer
from scripts.util.downstream import Average_gene_dynamics_whole_saveonly_single_trajectory_clinical

## Import Preprocessed Data and Define Dimensionality Reduction

- Please ensure you have run the `Preprocess_datasets.ipynb` notebook for the dataset you intend to use for downstream analysis.
- The preprocessed data should be saved in the `data/` folder with filenames ending in `_preprocessed.pkl`.
- When loading the preprocessed data below, make sure to use the correct and compatible filename corresponding to your dataset.
- We also load gene names from the gene expression matrix saved in the appropriate subfolder under the `data/` directory for each dataset. This step is optional and only needed if you wish to include **all genes** in the expression matrix. Otherwise, you can manually define a subset of genes of interest by assigning them to `gene_names`. For example, a gene subset for MCF7 cell line data might look like:  
  ```python
  gene_names = ['TEAD4', 'CDK2', 'IL6', 'RRM2', 'TGFB3', 'CDKN1A', 'MAP2K2']
- Each of the following cells corresponds to a specific dataset. Only run the cell for the dataset you're working with to avoid overwriting previously loaded data.


In [ ]:
## Loads MCF7 cell line data

data_name = "MCF7"

#loads the original data matrices
preprocessed_filename = data_dir + data_name + "_preprocessed.pkl"
with open(preprocessed_filename, "rb") as fr:
    try:
        time_label, full_matrix, projected_matrix, pca = pk.load(fr)
    except:
        time_label, full_matrix = pk.load(fr)   

pca = decomposition.PCA(n_components=2, random_state=0)
pca.fit(full_matrix)

d_red = 2
dimension_reduction = True

cls = set(time_label)
mats = {}
for c in cls:
    mats[c] = full_matrix[time_label == c].astype(float)

## Read Gene expression matrix (MCF7 cell line data)


# select genes from time series data
if os.path.exists(data_dir + '/MCF7 Cell Line/'+ "combined_matrix_transposed_basal_NDPR_Rgene.txt"):
    df_reduced_emt = pd.read_table(data_dir + '/MCF7 Cell Line/' + 'combined_matrix_transposed_basal_NDPR_Rgene.txt', sep="\t")
    print(df_reduced_emt)
    col_selection_emt = df_reduced_emt.columns[1:]
elif os.path.exists(data_dir + 'Gene_Expression_Matrix.txt'):
    col_selection_emt = np.loadtxt(data_dir + 'non_overlap_72_genes.txt', dtype=str)
    df_reduced_emt = df.filter(items=np.concatenate((['id'], col_selection_emt)))
    print(df_reduced_emt)
    print("%d EMT Hallmark genes are excluded!" % (len(col_selection_emt)-len(df_reduced_emt.columns[1:])))
    col_selection_emt = df_reduced_emt.columns[1:]
    df_reduced_emt.to_csv(data_dir + 'submatrix_72_genes_expression_matrix.txt', sep='\t', index=False)

  
print("Original space dimension = ", len(col_selection_emt))

# Get the list of gene names (excluding the first column which contains cell names)
gene_names = df_reduced_emt.columns[1:].tolist()

# Print the list of gene names
print(gene_names)  

In [ ]:
## Loads Patient PA3 data


data_name = "PA3"

#loads the original data matrices
preprocessed_filename = data_dir + data_name + "_preprocessed.pkl"
with open(preprocessed_filename, "rb") as fr:
    try:
        time_label, full_matrix, projected_matrix, pca = pk.load(fr)
    except:
        time_label, full_matrix = pk.load(fr)   

pca = decomposition.PCA(n_components=2, random_state=0)
pca.fit(full_matrix)

d_red = 2
dimension_reduction = True

cls = set(time_label)
mats = {}
for c in cls:
    mats[c] = full_matrix[time_label == c].astype(float)


## Read Gene expression matrix 
# select genes from time series data
if os.path.exists(data_dir + '/Patient_PA3/'+ "combined_matrix_transposed_palbo_BMC_nofibroblast_malignant_Rgene.txt"):
    df_reduced_emt = pd.read_table(data_dir + '/Patient_PA3/' + 'combined_matrix_transposed_palbo_BMC_nofibroblast_malignant_Rgene.txt', sep="\t")
    print(df_reduced_emt)
    col_selection_emt = df_reduced_emt.columns[1:]
elif os.path.exists(data_dir + 'Gene_Expression_Matrix.txt'):
    col_selection_emt = np.loadtxt(data_dir + 'non_overlap_72_genes.txt', dtype=str)
    df_reduced_emt = df.filter(items=np.concatenate((['id'], col_selection_emt)))
    print(df_reduced_emt)
    print("%d EMT Hallmark genes are excluded!" % (len(col_selection_emt)-len(df_reduced_emt.columns[1:])))
    col_selection_emt = df_reduced_emt.columns[1:]
    df_reduced_emt.to_csv(data_dir + 'submatrix_72_genes_expression_matrix.txt', sep='\t', index=False)

  
print("Original space dimension = ", len(col_selection_emt))

# Get the list of gene names (excluding the first column which contains cell names)
gene_names = df_reduced_emt.columns[1:].tolist()

# Print the list of gene names
print(gene_names)  


## Read Cell Day Matrix (MCF7 cell line data)

df_cls = pd.read_table(data_dir + '/Patient_PA3/' + 'Cells_Days_palbo_BMC_nofibroblast_malignant.txt', sep="\t")
print(df_cls)

cls = set(df_cls['day'])
print("classes:", cls)
print()

N_samples_cls = {}
for c in cls:
    N_samples_cls[c] = len(df_cls[df_cls['day']==c])
    print("Day ", c, ": sample size: ", N_samples_cls[c])

# Save the DataFrame with the new file name "Cells_Days.txt"
output_path = data_dir + "Cells_Days.txt"
df_cls.to_csv(output_path, sep="\t", index=False)
print(f"File saved as: {output_path}")

## Read Cell Day Matrix (MCF7 cell line data)

df_cls = pd.read_table(data_dir + '/MCF7 Cell Line/' + 'Cells_Days_basal_NDPR.txt', sep="\t")
print(df_cls)

cls = set(df_cls['day'])
print("classes:", cls)
print()

N_samples_cls = {}
for c in cls:
    N_samples_cls[c] = len(df_cls[df_cls['day']==c])
    print("Day ", c, ": sample size: ", N_samples_cls[c])

# Save the DataFrame with the new file name "Cells_Days.txt"
output_path = data_dir + "Cells_Days.txt"
df_cls.to_csv(output_path, sep="\t", index=False)
print(f"File saved as: {output_path}")

In [ ]:
## Loads Patient 862 data


data_name = "862"

#loads the original data matrices
preprocessed_filename = data_dir + data_name + "_preprocessed.pkl"
with open(preprocessed_filename, "rb") as fr:
    try:
        time_label, full_matrix, projected_matrix, pca = pk.load(fr)
    except:
        time_label, full_matrix = pk.load(fr)   

pca = decomposition.PCA(n_components=2, random_state=0)
pca.fit(full_matrix)

d_red = 2
dimension_reduction = True

cls = set(time_label)
mats = {}
for c in cls:
    mats[c] = full_matrix[time_label == c].astype(float)


## Read Gene expression matrix 
# select genes from time series data
if os.path.exists(data_dir + '/Patient_862/'+ "combined_matrix_transposed_palbo_NatMed_862_nofibroblast_malignant_Rgene.txt"):
    df_reduced_emt = pd.read_table(data_dir + '/Patient_862/' + 'combined_matrix_transposed_palbo_NatMed_862_nofibroblast_malignant_Rgene.txt', sep="\t")
    print(df_reduced_emt)
    col_selection_emt = df_reduced_emt.columns[1:]
elif os.path.exists(data_dir + 'Gene_Expression_Matrix.txt'):
    col_selection_emt = np.loadtxt(data_dir + 'non_overlap_72_genes.txt', dtype=str)
    df_reduced_emt = df.filter(items=np.concatenate((['id'], col_selection_emt)))
    print(df_reduced_emt)
    print("%d EMT Hallmark genes are excluded!" % (len(col_selection_emt)-len(df_reduced_emt.columns[1:])))
    col_selection_emt = df_reduced_emt.columns[1:]
    df_reduced_emt.to_csv(data_dir + 'submatrix_72_genes_expression_matrix.txt', sep='\t', index=False)

  
print("Original space dimension = ", len(col_selection_emt))

# Get the list of gene names (excluding the first column which contains cell names)
gene_names = df_reduced_emt.columns[1:].tolist()

# Print the list of gene names
print(gene_names)  


## Read Cell Day Matrix (MCF7 cell line data)

df_cls = pd.read_table(data_dir + '/Patient_862/' + 'cell_matrix_palbo_862_1.txt', sep="\t")
print(df_cls)

cls = set(df_cls['day'])
print("classes:", cls)
print()

N_samples_cls = {}
for c in cls:
    N_samples_cls[c] = len(df_cls[df_cls['day']==c])
    print("Day ", c, ": sample size: ", N_samples_cls[c])

# Save the DataFrame with the new file name "Cells_Days.txt"
output_path = data_dir + "Cells_Days.txt"
df_cls.to_csv(output_path, sep="\t", index=False)
print(f"File saved as: {output_path}")

In [ ]:
## Loads Patient 887 data


data_name = "887"

#loads the original data matrices
preprocessed_filename = data_dir + data_name + "_preprocessed.pkl"
with open(preprocessed_filename, "rb") as fr:
    try:
        time_label, full_matrix, projected_matrix, pca = pk.load(fr)
    except:
        time_label, full_matrix = pk.load(fr)   

pca = decomposition.PCA(n_components=2, random_state=0)
pca.fit(full_matrix)

d_red = 2
dimension_reduction = True

cls = set(time_label)
mats = {}
for c in cls:
    mats[c] = full_matrix[time_label == c].astype(float)


## Read Gene expression matrix 
# select genes from time series data
if os.path.exists(data_dir + '/Patient_887/'+ "combined_matrix_transposed_palbo_NatMed_887_nofibroblast_malignant_Rgene.txt"):
    df_reduced_emt = pd.read_table(data_dir + '/Patient_887/' + 'combined_matrix_transposed_palbo_NatMed_887_nofibroblast_malignant_Rgene.txt', sep="\t")
    print(df_reduced_emt)
    col_selection_emt = df_reduced_emt.columns[1:]
elif os.path.exists(data_dir + 'Gene_Expression_Matrix.txt'):
    col_selection_emt = np.loadtxt(data_dir + 'non_overlap_72_genes.txt', dtype=str)
    df_reduced_emt = df.filter(items=np.concatenate((['id'], col_selection_emt)))
    print(df_reduced_emt)
    print("%d EMT Hallmark genes are excluded!" % (len(col_selection_emt)-len(df_reduced_emt.columns[1:])))
    col_selection_emt = df_reduced_emt.columns[1:]
    df_reduced_emt.to_csv(data_dir + 'submatrix_72_genes_expression_matrix.txt', sep='\t', index=False)

  
print("Original space dimension = ", len(col_selection_emt))

# Get the list of gene names (excluding the first column which contains cell names)
gene_names = df_reduced_emt.columns[1:].tolist()

# Print the list of gene names
print(gene_names)  


## Read Cell Day Matrix (MCF7 cell line data)

df_cls = pd.read_table(data_dir + '/Patient_887/' + 'cell_matrix_palbo_887_1.txt', sep="\t")
print(df_cls)

cls = set(df_cls['day'])
print("classes:", cls)
print()

N_samples_cls = {}
for c in cls:
    N_samples_cls[c] = len(df_cls[df_cls['day']==c])
    print("Day ", c, ": sample size: ", N_samples_cls[c])

# Save the DataFrame with the new file name "Cells_Days.txt"
output_path = data_dir + "Cells_Days.txt"
df_cls.to_csv(output_path, sep="\t", index=False)
print(f"File saved as: {output_path}")

## Load Trajectory Inferred by PROFET

- Make sure the reconstructed cell trajectory (generated by PROFET after the force-matching step) is saved as a `.pickle` file.  
- The code below will load that `.pickle` file.  
- For example, in the case of MCF7 cell line data, the filename might be:
```
Palbo_NDPR_nofibroblast_malignant_Rgene_dim2-f_Lip=5e-2-t_size=50-network=64_64_64.pickle
 ```
- Be sure to replace this with the filename corresponding to your own dataset.
- Each of the following cells corresponds to a specific dataset. **Only run the cell for the dataset you're working with** to avoid overwriting previously loaded data.

In [ ]:
#loads the parameters and generates X1_trpts (MCF7 cell line data)

folder_name = 'Palbo_NDPR_nofibroblast_malignant_Rgene_dim2-f_Lip=5e-2-t_size=50-network=64_64_64' # mESC data pickle file name
result_dir = '%s/assets/Transport_genes/' % main_dir

if not os.path.exists(result_dir):
    os.makedirs(result_dir)
    print(f"Directory '{result_dir}' created.")

print("saving results to: ", result_dir)

d_red = 2
filename = result_dir + folder_name + ".pickle"
W, b, p = load_W(filename)

print("loaded: " + filename)
dt = p['numerical_ts'][-1]/200

X1_trpts = time_integration(pca.transform(mats[0]), T = p['numerical_ts'][-1], dt = dt)
physical_dt = dt * p['ts'][-1] / p['numerical_ts'][-1]

In [ ]:
#loads the parameters and generates X1_trpts (Patient PA3 data)

folder_name = 'Palbo_BMC_nofibroblast_malignant_Rgene_dim2-f_Lip=5e-2-t_size=50-network=64_64_64' # mESC data pickle file name
result_dir = '%s/assets/Transport_genes/' % main_dir

if not os.path.exists(result_dir):
    os.makedirs(result_dir)
    print(f"Directory '{result_dir}' created.")

print("saving results to: ", result_dir)

d_red = 2
filename = result_dir + folder_name + ".pickle"
W, b, p = load_W(filename)

print("loaded: " + filename)
dt = p['numerical_ts'][-1]/200

X1_trpts = time_integration(pca.transform(mats[0]), T = p['numerical_ts'][-1], dt = dt)
physical_dt = dt * p['ts'][-1] / p['numerical_ts'][-1]

In [ ]:
#loads the parameters and generates X1_trpts (Patient 862 data)

folder_name = 'Palbo_862_nofibroblast_malignant_Rgene_dim2-f_Lip=5e-2-t_size=50-network=64_64_64' # mESC data pickle file name
result_dir = '%s/assets/Transport_genes/' % main_dir

if not os.path.exists(result_dir):
    os.makedirs(result_dir)
    print(f"Directory '{result_dir}' created.")

print("saving results to: ", result_dir)

d_red = 2
filename = result_dir + folder_name + ".pickle"
W, b, p = load_W(filename)

print("loaded: " + filename)
dt = p['numerical_ts'][-1]/200

X1_trpts = time_integration(pca.transform(mats[0]), T = p['numerical_ts'][-1], dt = dt)
physical_dt = dt * p['ts'][-1] / p['numerical_ts'][-1]

In [ ]:
#loads the parameters and generates X1_trpts (Patient 887 data)

folder_name = 'Palbo_887_nofibroblast_malignant_Rgene_dim2-f_Lip=5e-2-t_size=50-network=64_64_64' # mESC data pickle file name
result_dir = '%s/assets/Transport_genes/' % main_dir

if not os.path.exists(result_dir):
    os.makedirs(result_dir)
    print(f"Directory '{result_dir}' created.")

print("saving results to: ", result_dir)

d_red = 2
filename = result_dir + folder_name + ".pickle"
W, b, p = load_W(filename)

print("loaded: " + filename)
dt = p['numerical_ts'][-1]/200

X1_trpts = time_integration(pca.transform(mats[0]), T = p['numerical_ts'][-1], dt = dt)
physical_dt = dt * p['ts'][-1] / p['numerical_ts'][-1]

## Subtrajectory analysis for breast cancer datasets (cell line and clinical data)
* To determine high versus low phenotypic change cell populations
* To make subtrajectory movies 
* To make subtrajectory static plots
* Save the csv files for the labels of subtrajectories

## Classify Single-Cell Trajectories Inferred from PROFET

- This section analyzes the **cell state displacements** from pre-treatment to post-treatment conditions based on the inferred PROFET trajectories.
- The output includes:
  - **Quantitative statistics** of trajectory lengths (i.e., phenotypic shifts)
  - **Histogram plots** of cell state displacements
- The histograms are used to **classify cells** into three trajectory groups:
  - **High**, **Medium**, and **Low** phenotypic shifts


In [ ]:
## finding the distribution plot of the cell transitions distances
exp_memo = folder_name

csv_output = f"{result_dir}{exp_memo}_X1_hat_displacement_stats.csv"
plot_output = f"{result_dir}{exp_memo}_X1_hat_displacement_distribution.png"
hist_output_path = f"{result_dir}{exp_memo}_X1_hat_displacement_histogram.csv"

X1_hat_labels = plot_X1_hat_displacement_distribution(X1_trpts, csv_output, plot_output, hist_output_path, exp_memo=exp_memo)

## Phenotypic Shift Classification and Visualization

This section classifies single-cell trajectories inferred from **PROFET** into **Low**, **Medium**, and **High** phenotypic shift groups, and visualizes them.

### 📥 Input
- Reconstructed trajectory (`.pickle`) from `PROFET_full_pipeline.ipynb`
- Displacement histogram plots (generated in the previous section)
- Output path for labels defined via `cluster_save_path`

### ⚙️ Parameters
- `source_t`: earliest time point of the trajectory
- `target_t`: latest time point of the trajectory
- `start_i`: start index for trajectory plotting
- `index`: temporal resolution (e.g., `1` = highest resolution)

### 📊 Output
- Static trajectory plots colored by phenotypic shift class (Low, Medium, High)
- CSV file with per-cell trajectory labels saved as:  
  `_X1_hat_deviation.csv`

### ⚠️ Notes
- For each dataset, use its corresponding function, such as:  
  `generate_static_cluster_plot_deviation_colormap_MCF7()`

- The main difference between these dataset-specific functions is how displacement thresholds are defined.  
  You must use the histogram generated in the previous section to decide thresholds.  
  Example (MCF7 cell line):
  ```python
  X1_hat_labels = np.full(displacements.shape, 'low', dtype=object)
  X1_hat_labels[(displacements > 3.017) & (displacements <= 3.853)] = 'medium'
  X1_hat_labels[displacements > 3.853] = 'high'

- **Important:** Each of the following cells corresponds to a specific dataset.  
  **Only run the cell for the dataset you're working with** to avoid overwriting previously loaded data.



In [ ]:
## MCF7 cell line data

source_t, target_t = 0, 4
optimal_k = 2
start_i = 0
index = 1
exp_memo = folder_name
output_file = f"{result_dir}{exp_memo}_static_celltypes_plot_deviation_colormap.png"

X1_hat_labels = generate_static_cluster_plot_deviation_colormap_MCF7(pca,
    source_t, target_t, start_i, X1_trpts, mats, index,intermediate_t=[ ], output_file = output_file
)

# Convert the labels into a DataFrame for saving
df_clusters = pd.DataFrame({
    "Cell_Index": np.arange(len(X1_hat_labels)),  # Assuming each cell has an index
    "Cluster_Label": X1_hat_labels
})

# Define the filename
cluster_save_path = f"{result_dir}{exp_memo}_X1_hat_deviation.csv"

# Save as CSV
df_clusters.to_csv(cluster_save_path, index=False)
print(f"Cluster labels saved to {cluster_save_path}")

# Return the saved filename for future use
cluster_save_path

In [ ]:
## Patient PA3

source_t, target_t = 0, 4
optimal_k = 2
start_i = 0
index = 1
exp_memo = folder_name
output_file = f"{result_dir}{exp_memo}_static_celltypes_plot_deviation_colormap.png"

X1_hat_labels = generate_static_cluster_plot_deviation_colormap_PA3(pca,
    source_t, target_t, start_i, X1_trpts, mats, index,intermediate_t=[ ], output_file = output_file
)

# Convert the labels into a DataFrame for saving
df_clusters = pd.DataFrame({
    "Cell_Index": np.arange(len(X1_hat_labels)),  # Assuming each cell has an index
    "Cluster_Label": X1_hat_labels
})

# Define the filename
cluster_save_path = f"{result_dir}{exp_memo}_X1_hat_deviation.csv"

# Save as CSV
df_clusters.to_csv(cluster_save_path, index=False)
print(f"Cluster labels saved to {cluster_save_path}")

# Return the saved filename for future use
cluster_save_path

In [ ]:
## Patient 862

source_t, target_t = 0, 4
optimal_k = 2
start_i = 0
index = 1
exp_memo = folder_name
output_file = f"{result_dir}{exp_memo}_static_celltypes_plot_deviation_colormap.png"

X1_hat_labels = generate_static_cluster_plot_deviation_colormap_862(pca,
    source_t, target_t, start_i, X1_trpts, mats, index,intermediate_t=[ ], output_file = output_file
)

# Convert the labels into a DataFrame for saving
df_clusters = pd.DataFrame({
    "Cell_Index": np.arange(len(X1_hat_labels)),  # Assuming each cell has an index
    "Cluster_Label": X1_hat_labels
})

# Define the filename
cluster_save_path = f"{result_dir}{exp_memo}_X1_hat_deviation.csv"

# Save as CSV
df_clusters.to_csv(cluster_save_path, index=False)
print(f"Cluster labels saved to {cluster_save_path}")

# Return the saved filename for future use
cluster_save_path

In [ ]:
## Patient 887

source_t, target_t = 0, 4
optimal_k = 2
start_i = 0
index = 1
exp_memo = folder_name
output_file = f"{result_dir}{exp_memo}_static_celltypes_plot_deviation_colormap.png"

X1_hat_labels = generate_static_cluster_plot_deviation_colormap_887(pca,
    source_t, target_t, start_i, X1_trpts, mats, index,intermediate_t=[ ], output_file = output_file
)

# Convert the labels into a DataFrame for saving
df_clusters = pd.DataFrame({
    "Cell_Index": np.arange(len(X1_hat_labels)),  # Assuming each cell has an index
    "Cluster_Label": X1_hat_labels
})

# Define the filename
cluster_save_path = f"{result_dir}{exp_memo}_X1_hat_deviation.csv"

# Save as CSV
df_clusters.to_csv(cluster_save_path, index=False)
print(f"Cluster labels saved to {cluster_save_path}")

# Return the saved filename for future use
cluster_save_path

## Save Cell Labels for Different Levels of Phenotypic Shifts

- This step is **optional** and is mainly used to label the **cell origins (pre-treatment states)** for each of the three trajectory shift classes (Low, Medium, High).
- The resulting file includes both the **phenotypic shift class** and the corresponding **cell IDs**.
- **Output file:** `_X1_hat_deviation_with_cell_ids.csv`


In [ ]:
## Save the labels at a specific time point for each class of subtrajectory (this is an example for time 0)

## Get all cell labels for each time point

df_cls_indexed = df_cls.set_index("id")

cell_ids_by_day = {}
for c in cls:
    idx = df_cls['day'] == c
    cell_ids_by_day[c] = df_cls_indexed.index[idx].tolist()  # extract index values for each day


# Step 1: Load saved file
df_loaded = pd.read_csv(cluster_save_path)

# Step 2: Verify lengths match
if len(df_loaded) != len(cell_ids_by_day[0]):
    raise ValueError("❌ Length mismatch: Cannot assign new cell IDs.")

# Step 3: Replace index or create new column
df_loaded["Cell_ID"] = cell_ids_by_day[0]  # Add as a column
df_loaded = df_loaded[["Cell_ID"] + [col for col in df_loaded.columns if col != "Cell_ID"]]  # Optional: move to front
df_loaded = df_loaded.drop(columns=["Cell_Index"])

# Step 4: Save to new file
new_path = cluster_save_path.replace(".csv", "_with_cell_ids.csv")
df_loaded.to_csv(new_path, index=False)
print(f"✅ Updated file saved to: {new_path}")

## Plot Single-Gene Dynamics for Every Cell

- This section generates **single-cell time courses** for each gene specified in `gene_of_interest`.
- In the example below, we use `gene_names`, which contains **all genes** from the gene expression matrix. You may instead provide a custom subset via `gene_of_interest`.
- For the **three patient datasets**, trajectories are highlighted by **phenotypic shift class** (Low / Medium / High) using the **same color scheme** used in the PCA visualization above, so that trajectory plots and PCA plots are visually consistent.
- Each of the following cells corresponds to a specific dataset. **Only run the cell for the dataset you’re working with** to avoid overwriting previously loaded data.


In [ ]:
## MCF7 cell line data

exp_memo = folder_name
gene_of_interest = gene_names #gene_names
cluster_save_path = f"{result_dir}{exp_memo}_X1_hat_deviation.csv"

source_t, target_t = 0, 4
start_i = 0
index = 1
intermediate_t = [4]
max_i = 200

# Iterate over each gene in the list
for gene in gene_of_interest:
    print(f"Processing gene: {gene}")
    try:
        # Call the function with the current gene
        subgroup_output_file = f"{result_dir}{exp_memo}_Celltypes_deviated_trajectories_violin_plot_{gene}.png"
        Average_gene_dynamics_whole_saveonly_single_trajectory_NDPR_breast_cancer(pca, gene_names, source_t, target_t, X1_trpts,mats, gene, index,p, max_i, intermediate_t = intermediate_t, subgroup_output_file = subgroup_output_file, cluster_save_path = cluster_save_path)
    except Exception as e:
        # Handle errors gracefully
        print(f"Error processing gene {gene}: {e}")



In [ ]:
## Patient data (PA3, 862, 887)

exp_memo = folder_name
gene_of_interest = gene_names #gene_names
cluster_save_path = f"{result_dir}{exp_memo}_X1_hat_deviation.csv"

source_t, target_t = 0, 4
start_i = 0
index = 1
intermediate_t = [4]
max_i = 200

# Iterate over each gene in the list
for gene in gene_of_interest:
    print(f"Processing gene: {gene}")
    try:
        # Call the function with the current gene
        subgroup_output_file = f"{result_dir}{exp_memo}_Celltypes_deviated_trajectories_violin_plot_{gene}.png"
        Average_gene_dynamics_whole_saveonly_single_trajectory_clinical(pca, gene_names, source_t, target_t, X1_trpts,mats, gene, index,p, max_i, intermediate_t = intermediate_t, subgroup_output_file = subgroup_output_file, cluster_save_path = cluster_save_path)
    except Exception as e:
        # Handle errors gracefully
        print(f"Error processing gene {gene}: {e}")


## Visualization of Histogram Plots of Cell State Displacement Using KDE

- This section visualizes the **Kernel Density Estimation (KDE)** plots of the **cell state displacements** computed in the earlier section.
- The **local minima** of the KDE curve are highlighted to suggest possible **thresholds** for classifying phenotypic shifts.
- Multiple datasets can be plotted together for **comparative analysis**.
- This visualization corresponds to **Figure 5B** in the manuscript.
 

In [ ]:
## Comparison of distributions across datasets (KDE)


import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
from scipy.signal import find_peaks
import os


#result_dir = '%s/assets/Transport_genes/' % main_dir
#output_dir = result_dir + 'output/'

# === Files ===
histogram_files = {
    "In vitro": "Palbo_NDPR_nofibroblast_malignant_Rgene_dim2-f_Lip=5e-2-t_size=50-network=64_64_64_X1_hat_displacement_histogram.csv",
    "PA3": "Palbo_BMC_nofibroblast_malignant_Rgene_dim2-f_Lip=5e-2-t_size=50-network=64_64_64_X1_hat_displacement_histogram.csv",
    "862": "Palbo_862_nofibroblast_malignant_Rgene_dim2-f_Lip=5e-2-t_size=50-network=64_64_64_X1_hat_displacement_histogram.csv",
    "887": "Palbo_887_nofibroblast_malignant_Rgene_dim2-f_Lip=5e-2-t_size=50-network=64_64_64_X1_hat_displacement_histogram.csv",
}

colors = {key: "gray" for key in histogram_files.keys()}
hist_data_kde = {}

# === Load and smooth ===
y_max_global = 0
for label, file in histogram_files.items():
    fpath = os.path.join(result_dir, file)
    if os.path.exists(fpath):
        df = pd.read_csv(fpath)
        data = np.repeat(df["bin_center"].values, df["count"].astype(int))
        kde = gaussian_kde(data, bw_method='scott')
        x_eval = np.linspace(min(data), max(data), 1000)
        density = kde(x_eval)
        hist_data_kde[label] = (x_eval, density)
        y_max_global = max(y_max_global, max(density))
    else:
        print(f"[Warning] File not found: {fpath}")

# === Plot ===
n = len(hist_data_kde)
fig, axes = plt.subplots(nrows=n, figsize=(8, 4 * n), sharex=True, constrained_layout=True)
if n == 1:
    axes = [axes]

for ax, (label, (x_eval, density)) in zip(axes, hist_data_kde.items()):
    color = colors.get(label, "gray")
    ax.plot(x_eval, density, color=color, lw=2)
    ax.fill_between(x_eval, density, alpha=0.3, color=color)

    # === Find valleys (local minima) ===
    valleys, _ = find_peaks(-density)
    x_vals = x_eval[valleys]
    y_vals = density[valleys]

    # Keep the 2 smallest minima in x-value
    sorted_idx = np.argsort(x_vals)
    top_two_idx = sorted_idx[:2]
    x_vals_top2 = x_vals[top_two_idx]
    y_vals_top2 = y_vals[top_two_idx]

    # === Plot only local minima ===
    ax.plot(x_vals_top2, y_vals_top2, "o", label="Local Minima", color = 'blue', markersize=10)
    for xv in x_vals_top2:
        ax.axvline(x=xv, color="black", linestyle="--", linewidth=1.5)

    # Print x-axis values of local minima
    print(f"{label} local minima x-values: {np.round(x_vals_top2, 3)}")

    ax.set_ylabel("Density", fontsize=23)
    ax.grid(alpha=0.3)
    ax.tick_params(axis="y", labelsize=23)
    ax.set_xlim(left=0)
    ax.set_ylim(top=y_max_global * 1.05)  # Ensure consistent y-limits across plots
    ax.set_title("", fontsize=23)

axes[-1].set_xlabel("Displacement (Euclidean Distance)", fontsize=24)
for ax in axes:
    ax.tick_params(axis="x", labelsize=22, which='both', labelbottom=True)

# === Save KDE figure ===
output_path = os.path.join(result_dir, "KDE_displacement_local_minima.pdf")
plt.savefig(output_path, dpi=300)
plt.show()
print(f"[✓] KDE plot saved to: {output_path}")

# === Save legend separately ===
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

legend_elements = [
    Line2D([0], [0], marker='o', color='blue', linestyle='None', label='Local Minimum', markersize=5),
    #Line2D([0], [0], color='black', linestyle='--', label='Threshold Cutoff')
]

fig_legend, ax_legend = plt.subplots(figsize=(4.5, 2))
ax_legend.axis("off")
legend = ax_legend.legend(handles=legend_elements, loc="center", fontsize=14)
legend_path = os.path.join(result_dir, "KDE_legend_only.pdf")
fig_legend.savefig(legend_path, bbox_inches='tight')
print(f"[✓] Legend saved to: {legend_path}")